# Ada v1 — neural upgrade (Colab-only training)

Self-contained (lab rule 11/12): all code inlined, no repo access, synthetic data only,
one config cell, train + eval + checkpoint/report download.

**What v1 adds over the v0 statistical tokenizer** (which already gives +43% bytes/tok):
a tiny differentiable byte front-end that *learns* boundaries instead of using corpus stats.
- **H-Net routing** (arXiv:2507.07955): boundary_prob = (1 - cos_sim(q h_t, k h_{t+1}))/2.
- **FSQ bottleneck** (arXiv:2309.15505): codebook-free per-channel rounding, no collapse.
- **MrT5 delete gate** (arXiv:2410.20771): learned mid-network deletion of redundant bytes.

v1 must beat the v0 ablation baseline (bytes/tok and downstream bits-per-byte) to ship.

In [ ]:
# === CONFIG CELL (rule 10: every knob here, no magic numbers below) ===
from dataclasses import dataclass, asdict
@dataclass
class V1Config:
    d_model: int = 128
    n_enc_layers: int = 2
    n_main_layers: int = 4
    n_heads: int = 4
    fsq_levels: tuple = (8, 8, 8, 5, 5, 5)   # implicit codebook = prod = 64000
    target_compression: float = 0.5          # MrT5 deletion target (keep ~50% of bytes)
    delete_reg_weight: float = 0.1
    ratio_loss_weight: float = 0.03           # H-Net boundary-ratio load balance
    seq_len: int = 512
    batch_size: int = 32
    lr: float = 3e-4
    steps: int = 2000
    seed: int = 0
    run_name: str = 'ada_v1_r1'
cfg = V1Config()
print(asdict(cfg))

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, random
torch.manual_seed(cfg.seed); random.seed(cfg.seed)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def round_ste(z):
    return z + (torch.round(z) - z).detach()

class FSQ(nn.Module):  # arXiv:2309.15505, faithful to Cosmos best/cosmos-tokenizer quantizers.py
    def __init__(self, levels):
        super().__init__(); self.register_buffer('L', torch.tensor(levels))
        self.dim = len(levels)
    def forward(self, z):
        half = (self.L - 1) * 0.5
        zhat = round_ste(torch.tanh(z) * half) / half
        return zhat

class Router(nn.Module):  # H-Net dynamic chunking, best/hnet/hnet/modules/dc.py:69
    def __init__(self, d):
        super().__init__(); self.q = nn.Linear(d, d, bias=False); self.k = nn.Linear(d, d, bias=False)
        with torch.no_grad(): self.q.weight.copy_(torch.eye(d)); self.k.weight.copy_(torch.eye(d))
    def forward(self, h):
        cos = F.cosine_similarity(self.q(h[:, :-1]), self.k(h[:, 1:]), dim=-1)
        p = ((1 - cos) / 2).clamp(0, 1)
        p = F.pad(p, (1, 0), value=1.0)  # first position is always a boundary
        return p

class DeleteGate(nn.Module):  # MrT5 ScaledSigmoid, best/mrt5/models/modeling_mrt5.py:104
    def __init__(self, d, scale=-30.0):
        super().__init__(); self.ff = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1)); self.scale = scale
    def forward(self, h):
        logit = self.ff(h).squeeze(-1)
        return self.scale * torch.sigmoid(-logit)  # ~0 keep, large-negative -> masked in attn

class AdaV1(nn.Module):
    def __init__(self, c):
        super().__init__(); d = c.d_model
        self.emb = nn.Embedding(320, d)
        enc = nn.TransformerEncoderLayer(d, c.n_heads, 4*d, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc, c.n_enc_layers)
        self.router = Router(d); self.gate = DeleteGate(d)
        self.proj_in = nn.Linear(d, len(c.fsq_levels)); self.fsq = FSQ(c.fsq_levels)
        self.proj_out = nn.Linear(len(c.fsq_levels), d)
        main = nn.TransformerEncoderLayer(d, c.n_heads, 4*d, batch_first=True, activation='gelu')
        self.main = nn.TransformerEncoder(main, c.n_main_layers)
        self.head = nn.Linear(d, 320)
    def forward(self, x):
        h = self.encoder(self.emb(x))
        bprob = self.router(h)                       # (B,L) boundary probabilities
        gate = self.gate(h)                          # (B,L) delete-gate mask
        z = self.proj_out(self.fsq(self.proj_in(h))) # FSQ bottleneck
        h2 = self.main(z + h)
        return self.head(h2), bprob, gate

In [ ]:
# Synthetic byte data (rule 11: no downloads). Replace with a real corpus stream on Colab.
VOCAB=('the of and to in a is that it for on with model token byte patch entropy merge '
       'vocab compression tokenizer boundary dynamic chunk of the in the by the way').split()
def sample_line(rng):
    return ' '.join(rng.choice(VOCAB) for _ in range(rng.randint(20,60)))
def batch(c, rng):
    rows=[]
    for _ in range(c.batch_size):
        b=sample_line(rng).encode()[:c.seq_len]; b=b+bytes(c.seq_len-len(b)); rows.append(list(b))
    return torch.tensor(rows, device=dev)

In [ ]:
# Train. Loss = next-byte CE + MrT5 deletion-rate reg + H-Net boundary-ratio load balance.
import math
rng=random.Random(cfg.seed); model=AdaV1(cfg).to(dev); opt=torch.optim.AdamW(model.parameters(), lr=cfg.lr)
log=[]
for step in range(cfg.steps):
    x=batch(cfg,rng); logits,bprob,gate=model(x)
    ce=F.cross_entropy(logits[:,:-1].reshape(-1,320), x[:,1:].reshape(-1))
    keep_rate=(gate > -1.0).float().mean()                         # gate near 0 => byte kept
    del_reg=cfg.delete_reg_weight*((keep_rate-cfg.target_compression)**2)  # MrT5 target rate
    ratio=cfg.ratio_loss_weight*((bprob.mean()-cfg.target_compression)**2) # H-Net ratio balance
    loss=ce+del_reg+ratio; opt.zero_grad(); loss.backward(); opt.step()
    if step%200==0:
        line=f'step {step} ce {ce.item():.3f} bpb {ce.item()/math.log(2):.3f} keep {keep_rate.item():.2f}'
        print(line); log.append(line)

In [ ]:
# Save checkpoint + run report, then download both (rule 12: two artifacts per run).
import json, zipfile
torch.save(model.state_dict(), f'{cfg.run_name}.pt')
report=f'# {cfg.run_name} report\n\nconfig: {json.dumps(asdict(cfg))}\n\n' + '\n'.join(log)
open(f'{cfg.run_name}_report.md','w').write(report)
with zipfile.ZipFile(f'{cfg.run_name}.zip','w') as z: z.write(f'{cfg.run_name}.pt')
try:
    from google.colab import files
    files.download(f'{cfg.run_name}.zip'); files.download(f'{cfg.run_name}_report.md')
except Exception as e:
    print('not on colab:', e)